# Project 22 — Query Rewriting RAG Lab

## Rewrite Vague Questions Before Retrieval for Better Results

**Goal:** Compare four query-rewriting strategies — simple clarification,
HyDE (Hypothetical Document Embeddings), multi-query expansion, and
step-back prompting — and measure how each improves retrieval relevance
over a baseline of no rewriting.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

```
Vague query ──► Rewriter (LLM) ──► Improved query ──► Retriever ──► LLM ──► Answer
                   │                                      │
           Strategy choices:                        ChromaDB
           • Clarify                               (dense search)
           • HyDE
           • Multi-query
           • Step-back
```

### What You'll Learn

1. Why vague queries cause retrieval failures
2. Four distinct query-rewriting strategies
3. Side-by-side retrieval quality comparison
4. End-to-end rewriting RAG pipeline
5. When rewriting helps vs when it hurts

### Prerequisites

- **Ollama** running locally with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")

## Step 2 — Build Knowledge Base

We create a small but realistic knowledge base. The documents use
proper technical language, but the *test queries* will be vague,
informal, or ambiguous — exactly the kind users type in practice.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

docs = [
    Document(page_content="LangChain is a framework for building LLM applications. "
        "It provides chains, agents, retrieval components, and memory modules.",
        metadata={"topic": "langchain", "id": 1}),
    Document(page_content="ChromaDB is an open-source vector database for AI applications. "
        "It supports in-memory and persistent storage with metadata filtering.",
        metadata={"topic": "chroma", "id": 2}),
    Document(page_content="Ollama runs large language models locally on your machine. "
        "It supports Llama, Mistral, Qwen, Gemma, and many other model families.",
        metadata={"topic": "ollama", "id": 3}),
    Document(page_content="RAG retrieval quality depends on chunk size, overlap, embedding "
        "model choice, and the retrieval algorithm used (BM25, dense, hybrid).",
        metadata={"topic": "rag_quality", "id": 4}),
    Document(page_content="Query rewriting transforms user queries into forms better suited "
        "for retrieval. Techniques include HyDE, step-back, and multi-query expansion.",
        metadata={"topic": "rewriting", "id": 5}),
    Document(page_content="Sentence embeddings map text to dense vectors in a shared space. "
        "Cosine similarity measures how semantically close two pieces of text are.",
        metadata={"topic": "embeddings", "id": 6}),
    Document(page_content="Prompt engineering improves LLM outputs through careful instruction "
        "design, few-shot examples, chain-of-thought, and system messages.",
        metadata={"topic": "prompting", "id": 7}),
    Document(page_content="Fine-tuning adapts pre-trained models to specific domains using "
        "supervised training on smaller datasets. LoRA reduces GPU memory needs.",
        metadata={"topic": "finetuning", "id": 8}),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="rewrite_lab")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"✅ Knowledge base indexed: {len(docs)} documents")

## Step 3 — Baseline Retrieval (No Rewriting)

These vague queries simulate real user behaviour — informal, ambiguous,
or missing key terms. Let's see how the retriever handles them raw.

In [ ]:
vague_queries = [
    ("how do I make it work better?",       "rag_quality",  "Vague"),
    ("that vector thing",                    "chroma",       "Ambiguous"),
    ("what runs models on my laptop?",       "ollama",       "Informal"),
    ("the python library for LLM apps",      "langchain",    "Imprecise"),
    ("make my prompts better",               "prompting",    "Vague"),
    ("how do I search text by meaning?",     "embeddings",   "Semantic"),
]

print("=" * 65)
print("BASELINE — No Rewriting")
print("=" * 65)
baseline_hits = 0
for q, expected, qtype in vague_queries:
    results = retriever.invoke(q)
    top_topic = results[0].metadata["topic"] if results else "none"
    hit = "✓" if top_topic == expected else "✗"
    if top_topic == expected:
        baseline_hits += 1
    print(f"  {hit} [{qtype:>9}] {q!r:<42} → {top_topic}")
print(f"\nBaseline Hit@1: {baseline_hits}/{len(vague_queries)}")

## Step 4 — Define Four Rewriting Strategies

Each strategy transforms the vague query before retrieval:

| Strategy | Idea |
|---|---|
| **Clarify** | Rewrite into a clear, specific search query |
| **HyDE** | Generate a hypothetical answer, then use it as search query |
| **Multi-query** | Expand into 3 query variants, retrieve from all |
| **Step-back** | Ask a more general question to get broader context |

In [ ]:
# Strategy 1: Simple clarification
clarify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite this vague query into a clear, specific search query. "
     "Add context that makes the query unambiguous. Return ONLY the rewritten query, no explanation."),
    ("human", "{query}")
])
clarify_chain = clarify_prompt | llm | StrOutputParser()

# Strategy 2: HyDE — Hypothetical Document Embeddings
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a short technical paragraph that would be the ideal answer to this "
     "question. Write as if you are a documentation page, not a conversational answer. "
     "Return ONLY the paragraph."),
    ("human", "{query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

# Strategy 3: Multi-query expansion
multi_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 3 different search queries that could help answer this question. "
     "Each should approach the topic from a different angle. "
     "Return one query per line, no numbering or bullets."),
    ("human", "{query}")
])
multi_chain = multi_prompt | llm | StrOutputParser()

# Strategy 4: Step-back prompting
stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a broader, more general question that would help answer the "
     "specific question below. Return ONLY the step-back question."),
    ("human", "{query}")
])
stepback_chain = stepback_prompt | llm | StrOutputParser()

print("✅ Four rewriting strategies defined")

## Step 5 — Compare All Strategies Side-by-Side

For each vague query we apply all four strategies, then check if
the top retrieved document matches the expected topic.

In [ ]:
def evaluate_strategy(name, rewrite_fn, queries):
    """Evaluate a rewriting strategy on all test queries."""
    hits = 0
    for q, expected, _ in queries:
        try:
            rewritten = rewrite_fn(q)
            if isinstance(rewritten, list):
                # Multi-query: retrieve from all variants
                all_topics = set()
                for sq in rewritten[:3]:
                    for r in retriever.invoke(sq):
                        all_topics.add(r.metadata["topic"])
                hit = expected in all_topics
            else:
                results = vectorstore.similarity_search(rewritten, k=2)
                hit = results[0].metadata["topic"] == expected if results else False
        except Exception:
            hit = False
        if hit:
            hits += 1
    return hits / len(queries)

def clarify_fn(q):
    return clarify_chain.invoke({"query": q})

def hyde_fn(q):
    return hyde_chain.invoke({"query": q})

def multi_fn(q):
    result = multi_chain.invoke({"query": q})
    return [sq.strip() for sq in result.strip().split("\n") if sq.strip()]

def stepback_fn(q):
    return stepback_chain.invoke({"query": q})

print("Evaluating strategies (this calls the LLM for each query)...\n")

strategies = [
    ("Baseline",  lambda q: q),
    ("Clarify",   clarify_fn),
    ("HyDE",      hyde_fn),
    ("Multi-Q",   multi_fn),
    ("Step-back", stepback_fn),
]

print(f"{'Strategy':<12} {'Hit@1':>8}")
print("-" * 22)
for name, fn in strategies:
    acc = evaluate_strategy(name, fn, vague_queries)
    print(f"{name:<12} {acc:>7.0%}")

## Step 6 — Detailed Rewrite Inspection

Let's look at what each strategy *actually* produces for a few
queries. This builds intuition for when each approach helps.

In [ ]:
inspect_queries = [
    "how do I make it work better?",
    "that vector thing",
    "make my prompts better",
]

for q in inspect_queries:
    print(f"\nOriginal: {q!r}")
    print(f"  Clarify:   {clarify_chain.invoke({'query': q})!r}")
    print(f"  HyDE:      {hyde_chain.invoke({'query': q})[:80]}...")
    multi = multi_chain.invoke({"query": q})
    subs = [s.strip() for s in multi.strip().split("\n") if s.strip()][:3]
    print(f"  Multi-Q:   {subs}")
    print(f"  Step-back: {stepback_chain.invoke({'query': q})!r}")

## Step 7 — End-to-End Rewriting RAG Pipeline

Complete pipeline: rewrite → retrieve → generate answer.
We compare baseline vs the best rewriting strategy.

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context. Be concise."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])
answer_chain = qa_prompt | llm | StrOutputParser()

def rewriting_rag(query, strategy="clarify"):
    """Full pipeline: rewrite → retrieve → answer."""
    if strategy == "clarify":
        rewritten = clarify_chain.invoke({"query": query})
    elif strategy == "hyde":
        rewritten = hyde_chain.invoke({"query": query})
    elif strategy == "stepback":
        rewritten = stepback_chain.invoke({"query": query})
    else:
        rewritten = query

    retrieved = retriever.invoke(rewritten)
    context = "\n".join([d.page_content for d in retrieved])
    answer = answer_chain.invoke({"context": context, "question": query})
    sources = [d.metadata["topic"] for d in retrieved]
    return {"rewritten": rewritten, "answer": answer, "sources": sources}

demo = [
    "how do I make retrieval better?",
    "what runs LLMs locally?",
]

for q in demo:
    print(f"\nQ: {q}")
    for strat in ["none", "clarify", "hyde"]:
        r = rewriting_rag(q, strategy=strat)
        label = strat.upper() if strat != "none" else "BASELINE"
        print(f"  [{label}] Sources: {r['sources']} — {str(r['answer'])[:100]}...")

## Step 8 — When Rewriting Hurts

Not all queries benefit from rewriting. Precise, well-formed queries can
get *worse* after rewriting if the LLM changes the intent. Let's test.

In [ ]:
precise_queries = [
    "What is ChromaDB?",
    "How does BM25 work?",
    "What is LoRA fine-tuning?",
]

print("Precise queries — does rewriting help or hurt?\n")
for q in precise_queries:
    baseline_r = retriever.invoke(q)
    baseline_top = baseline_r[0].metadata["topic"] if baseline_r else "none"

    rewritten = clarify_chain.invoke({"query": q})
    rewrite_r = vectorstore.similarity_search(rewritten, k=2)
    rewrite_top = rewrite_r[0].metadata["topic"] if rewrite_r else "none"

    status = "SAME" if baseline_top == rewrite_top else "CHANGED"
    print(f"  Q: {q}")
    print(f"    Baseline: {baseline_top} | Rewritten: {rewrite_top} [{status}]")

## Limitations and Tradeoffs

1. **Latency:** Every rewriting strategy adds an LLM call before retrieval,
   roughly doubling response time.

2. **Small knowledge base:** With only 8 documents, differences are
   subtle. Larger corpora show clearer gains.

3. **Strategy selection:** No single strategy wins for all query types.
   Production systems may need a router to pick the best strategy.

4. **HyDE hallucination risk:** The hypothetical document may contain
   wrong information that biases retrieval toward incorrect results.

5. **Multi-query cost:** Generating and retrieving from 3+ queries
   multiplies both LLM and retrieval costs.

## How to Extend This Project

1. **Query classifier:** Build a classifier that routes queries to the
   best rewriting strategy based on vagueness score.

2. **DSPy optimization:** Use DSPy to automatically optimize the
   rewriting prompts based on retrieval metrics.

3. **Combined strategies:** Chain step-back + clarify for two-stage
   rewriting on very ambiguous queries.

4. **A/B testing framework:** Log both rewritten and original queries
   and compare downstream answer quality.

## What You Learned

| Concept | What We Did |
|---|---|
| **Query clarification** | LLM rewrites vague queries into specific ones |
| **HyDE** | Generate hypothetical answer, use as search query |
| **Multi-query expansion** | Multiple query variants for wider recall |
| **Step-back prompting** | Generalize query for broader context |
| **Baseline comparison** | Measured Hit@1 with and without rewriting |
| **Failure analysis** | When rewriting hurts precise queries |
| **End-to-end RAG** | Complete rewrite → retrieve → generate pipeline |